# Orientation-aware thin structure segmentation (public data)

Compare the same compact convolutional segmenter on intensity alone and intensity plus six steerable second-derivative channels. Synthesize responses at six angles from Gaussian Hessian components. Both models use the same mask loss and best-validation-Dice epoch selection. The design adapts the orientation-aware detection idea in Li and Barbu. It uses fixed steerable filters and a conventional CNN. Use the public DRIVE retinal fundus data with vessel labels. Train on 16 images and validate on four from the original training partition. Evaluate on 20 separate test images at 144×144. Retinal vessels are thin structures and serve as an accessible analogue for guidewires.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'steerable-thin-structures-real-data-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0', 'torch==2.14.1', 'Pillow==12.3.0']],check=True)

In [ ]:
(root / 'prepare.py').write_text('from concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nfrom urllib.request import urlopen\nimport os\nimport time\n\nROOT = (\n    Path(os.environ.get("PORTFOLIO_DATA", Path(__file__).resolve().parent / "data"))\n    / "drive"\n)\nREPO = "https://huggingface.co/datasets/Zomba/DRIVE-digital-retinal-images-for-vessel-extraction/resolve/e43e3f8a4b1146db231648786912d0ac882067b2/"\n\n\ndef obtain(name):\n    target = ROOT / name\n    if target.exists() and target.stat().st_size > 5000:\n        return\n    target.parent.mkdir(parents=True, exist_ok=True)\n    for attempt in range(3):\n        try:\n            with urlopen(REPO + name, timeout=35) as reply:\n                data = reply.read()\n            if len(data) < 5000:\n                raise ValueError("Incomplete image " + name)\n            temp = target.with_suffix(".part")\n            temp.write_bytes(data)\n            temp.replace(target)\n            return\n        except (OSError, ValueError):\n            if attempt == 2:\n                raise\n            time.sleep(2)\n\n\nif __name__ == "__main__":\n    files = [\n        f"{split}/{kind}/{i:02}{\'_manual1\' if split == \'val\' and kind == \'label\' else \'\'}.{ext}"\n        for split, ids in [("train", range(21, 41)), ("val", range(1, 21))]\n        for i in ids\n        for kind, ext in [("input", "tif"), ("label", "png")]\n    ]\n    with ThreadPoolExecutor(8) as pool:\n        list(pool.map(obtain, files))\n',encoding='utf8')
subprocess.run([sys.executable,'prepare.py'],cwd=root,check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.metrics import (\n    average_precision_score,\n    roc_auc_score,\n)\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\nfrom PIL import Image\n\n\ndef dataset(seed):\n    images = []\n    masks = []\n    for split, ids in [("train", range(21, 41)), ("val", range(1, 21))]:\n        for i in ids:\n            root = DATA / "drive" / split\n            with Image.open(root / "input" / f"{i:02}.tif") as im:\n                channel = im.convert("RGB").getchannel("G")\n                images.append(np.asarray(channel.resize((144, 144))) / 255)\n            with Image.open(\n                root / "label" / f"{i:02}{\'_manual1\' if split == \'val\' else \'\'}.png"\n            ) as im:\n                arr = np.asarray(\n                    im.convert("L").resize((144, 144), Image.Resampling.NEAREST)\n                )\n                masks.append(arr > 0 if arr.max() == 1 else arr > 127)\n    assert all(0.01 < a.mean() < 0.3 for a in masks)\n    tr, va = train_test_split(np.arange(20), test_size=0.2, random_state=seed)\n    return np.asarray(images)[..., None], np.asarray(masks), tr, va, np.arange(20, 40)\n\n\nfrom scipy.ndimage import gaussian_filter\n\n\ndef bank(images):\n    x = images[:, 0].numpy().astype("float32")\n    xx = gaussian_filter(x, (0, 1.15, 1.15), order=(0, 0, 2))\n    xy = gaussian_filter(x, (0, 1.15, 1.15), order=(0, 1, 1))\n    yy = gaussian_filter(x, (0, 1.15, 1.15), order=(0, 2, 0))\n    angles = np.arange(6) * np.pi / 6\n    responses = [\n        (np.cos(a) ** 2 * xx + 2 * np.cos(a) * np.sin(a) * xy + np.sin(a) ** 2 * yy)\n        for a in angles\n    ]\n    return torch.cat((images, torch.from_numpy(np.stack(responses, axis=1).astype(np.float32) * 8)), 1)\n\n\nimport torch\nfrom sklearn.model_selection import train_test_split\nfrom torch import nn\nfrom torch.nn import functional as F\n\ntorch.set_num_threads(2)\n\n\nclass Net(nn.Module):\n    def __init__(self, channels):\n        super().__init__()\n        self.stem = nn.Sequential(nn.Conv2d(channels, 12, 3, padding=1), nn.ReLU())\n        self.context = nn.Sequential(\n            nn.MaxPool2d(2),\n            nn.Conv2d(12, 20, 3, padding=1),\n            nn.ReLU(),\n            nn.Conv2d(20, 20, 3, padding=1),\n            nn.ReLU(),\n        )\n        self.mask = nn.Sequential(\n            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 1, 1)\n        )\n        self.classifier = nn.Linear(20, 1)\n\n    def forward(self, x):\n        low = self.stem(x)\n        z = self.context(low)\n        up = F.interpolate(z, size=low.shape[-2:], mode="bilinear", align_corners=False)\n        return self.mask(torch.cat((low, up), 1)), self.classifier(\n            z.mean((2, 3))\n        ).flatten()\n\n\ndef accuracy(logits, y):\n    mask = logits >= 0\n    overlap = (mask * y).sum((1, 2, 3))\n    return (\n        ((2 * overlap + 1) / (mask.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1)).mean().item()\n    )\n\n\ndef run(seed):\n    images, masks, tr, va, te = dataset(seed)\n    images = torch.tensor(images, dtype=torch.float32).permute(0, 3, 1, 2)\n    masks = torch.tensor(masks, dtype=torch.float32)[:, None]\n    areas = masks.mean((1, 2, 3))\n    threshold = areas[tr].median()\n    category = (areas > threshold).float()\n    schemes = [("intensity", images, False), ("oriented", bank(images), False)]\n    out = {}\n    for name, features, aux in schemes:\n        torch.manual_seed(seed)\n        model = Net(features.shape[1])\n        opt = torch.optim.AdamW(model.parameters(), lr=0.003)\n        best = -1\n        for _ in range(22):\n            model.train()\n            for batch in torch.randperm(len(tr)).split(16):\n                ids = tr[batch.numpy()]\n                p, c = model(features[ids])\n                y = masks[ids]\n                soft = p.sigmoid()\n                overlap = (2 * (soft * y).sum() + 1) / (soft.sum() + y.sum() + 1)\n                weight = (1 - y.mean()) / y.mean().clamp(min=0.02)\n                loss = (\n                    F.binary_cross_entropy_with_logits(\n                        p, y, pos_weight=weight.clamp(max=12)\n                    )\n                    + 1\n                    - overlap\n                )\n                if aux:\n                    loss += 0.45 * F.binary_cross_entropy_with_logits(c, category[ids])\n                opt.zero_grad()\n                loss.backward()\n                opt.step()\n            model.eval()\n            with torch.no_grad():\n                val = accuracy(model(features[va])[0], masks[va])\n            if val > best:\n                best = val\n                state = {k: v.detach().clone() for k, v in model.state_dict().items()}\n        model.load_state_dict(state)\n        folder = ROOT / "checkpoints"\n        folder.mkdir(exist_ok=True)\n        torch.save(state, folder / f"{name}-{seed}.pt")\n        with torch.no_grad():\n            pred, logits = model(features[te])\n            result = {"dice": accuracy(pred, masks[te]), "test_images": len(te)}\n            result["mask_prevalence"] = masks[te].mean().item()\n        out[name] = result\n    return out\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())